<a href="https://colab.research.google.com/github/casadesoftware/Color/blob/main/Copia_de_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar el modelo que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [8]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv


name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga del modelo.

In [9]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print("Ollama instalado.")

Ollama instalado.


In [10]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.2


### **DESCARGAR EL MODELO**

Vamos a trabajar con **Llama 3.1 de 8B**. Guardamos el nombre en la variable `MODELO` para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [11]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)
MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [12]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [17]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'Es improbable que las cosas no resulten como estaban planeadas' Respuesta muy breve y corta."

response_zero = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.message.content)

Zero-shot: La respuesta es: Positivo.


In [20]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "Me esforcé mucho pero la calificación fue inmejorable."
Sentimiento: """

response_few = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: Sentimiento: Positivo


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [21]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

¡Vamos a resolverlo paso a paso!

1. El primer tren sale de la ciudad A a una velocidad de 80 km/h y sale 2 horas antes del segundo tren. En ese tiempo, el primer tren cubre una distancia de:

Distancia = Velocidad x Tiempo = 80 km/h x 2 horas = 160 km

2. El segundo tren sale de la ciudad A a una velocidad de 120 km/h. Debemos encontrar la diferencia de velocidad entre los dos trenes:

Diferencia de velocidad = Velocidad del segundo tren - Velocidad del primer tren
= 120 km/h - 80 km/h
= 40 km/h

3. El primer tren tiene una ventaja de 160 km. Para que el segundo tren lo alcance, debe cubrir esta distancia a una velocidad de 40 km/h más que el primer tren. Entonces, la velocidad efectiva con la que el segundo tren se acerca al primero es:

Velocidad efectiva = Velocidad del primer tren + Diferencia de velocidad
= 80 km/h + 40 km/h
= 120 km/h

4. Ahora, calculemos el tiempo necesario para que el segundo tren alcance al primero:

Tiempo = Distancia / Velocidad
= 160 km / 120 km/h
= 1,33 

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [22]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
response_alucinacion = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": "Quien ganó las elecciones presidenciales en Brasil en 2026? Respuesta muy breve y corta."}]
)

print(response_alucinacion.message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

Lo siento, pero no tengo información disponible sobre las elecciones presidenciales en Brasil en 2026, ya que la información sobre eventos futuros no está disponible a la fecha de mi última actualización en abril de 2023.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [23]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [24]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [33]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo comprar otra prenda igual?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su empaque original.


In [31]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat(
    model=MODELO,
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

Lo siento, nuestra política no cubre la devolución de productos que no fueron comprados en nuestra tienda.


## **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [34]:
# Verificar que Colab asignó una GPU
!nvidia-smi

Sun Oct 11 03:53:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             31W /   70W |    5767MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [36]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [37]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

Servidor de Ollama listo.


In [38]:
# Descargar el modelo Llama e inicializar el cliente
!ollama pull {"llama3.1:8b"}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


**2. Define tu base de conocimiento y genera embeddings:**

* Instala e importa la librería de  `sentence-transformers` junto con `numpy`.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings.

In [47]:
# Instalar e importar librerías
!pip install -q sentence-transformers numpy
import numpy as np
import requests
from sentence_transformers import SentenceTransformer, util

In [48]:
# Definir la lista documentos y generar sus embeddings
documentos = [
    "Política de Privacidad: Todos los datos personales recopilados serán encriptados y almacenados en servidores seguros, cumpliendo con la normativa vigente de protección de datos.",
    "Términos de Servicio: El usuario es responsable de mantener la confidencialidad de su contraseña. Cualquier actividad realizada bajo su cuenta se considerará de su absoluta responsabilidad.",
    "Política de Reembolso: Las solicitudes de devolución deberán tramitarse a través de soporte técnico dentro de los primeros 15 días naturales posteriores a la fecha de compra original."
]
modelo = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = modelo.encode(documentos)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [49]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    pregunta_embedding = modelo.encode(pregunta)
    similitudes = util.cos_sim(pregunta_embedding, embeddings)
    indice_max = np.argmax(similitudes)
    mejor_fragmento = documentos[indice_max]
    puntaje = similitudes[0][indice_max].item()

    return mejor_fragmento, puntaje

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [55]:
pregunta = "¿Cuánto tiempo tengo para solicitar un reembolso y cómo debo hacerlo?"

payload_sin_rag = {
    "model": "llama3.1:8b",
    "prompt": pregunta,
    "stream": False
}

respuesta = requests.post("http://localhost:11434/api/generate", json=payload_sin_rag)
datos_json = respuesta.json()

if "response" in datos_json:
    respuesta_sin_rag = datos_json["response"]
    print("Pregunta:", pregunta)
    print("-" * 50)
    print("Respuesta SIN RAG:\n")
    print(respuesta_sin_rag)
else:
    print("Ocurrió un error con Ollama. Respuesta del servidor:")
    print(datos_json)

Pregunta: ¿Cuánto tiempo tengo para solicitar un reembolso y cómo debo hacerlo?
--------------------------------------------------
Respuesta SIN RAG:

Puedes solicitar un reembolso en un plazo máximo de 30 días a partir de la fecha de emisión de la factura. Puedes hacerlo a través de nuestra página web o a través del servicio de atención al cliente del establecimiento en el que realizaste la compra.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [56]:
pregunta = "¿Cuánto tiempo tengo para solicitar un reembolso y cómo debo hacerlo?"

contexto, confianza = buscar_fragmento(pregunta)

prompt_aumentado = f"""Responde a la pregunta basándote ÚNICAMENTE en el siguiente contexto.
Si la respuesta no está en el contexto, di que no tienes esa información.

Contexto:
{contexto}

Pregunta:
{pregunta}
"""

payload_con_rag = {
    "model": "llama3.1:8b",
    "prompt": prompt_aumentado,
    "stream": False
}

respuesta = requests.post("http://localhost:11434/api/generate", json=payload_con_rag)
datos_json = respuesta.json()

if "response" in datos_json:
    respuesta_con_rag = datos_json["response"]

    print(f"Contexto inyectado (Confianza: {confianza:.2f}):\n{contexto}\n")
    print("-" * 50)
    print("Respuesta CON RAG:\n")
    print(respuesta_con_rag)
else:
    print("Ocurrió un error con Ollama:", datos_json)

Contexto inyectado (Confianza: 0.52):
Política de Reembolso: Las solicitudes de devolución deberán tramitarse a través de soporte técnico dentro de los primeros 15 días naturales posteriores a la fecha de compra original.

--------------------------------------------------
Respuesta CON RAG:

Tienes 15 días naturales posteriores a la fecha de compra original para solicitar un reembolso y debes tramitarlo a través de soporte técnico.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [40]:
# Mostrar ambas respuestas para comparar
Respuesta SIN RAG: Puedes solicitar un reembolso en un plazo máximo de 30 días a partir de la fecha de emisión de la factura. Puedes hacerlo a través de nuestra página web o a través del servicio de atención al cliente del establecimiento en el que realizaste la compra.

Respuesta CON RAG: Tienes 15 días naturales posteriores a la fecha de compra original para solicitar un reembolso y debes tramitarlo a través de soporte técnico.


Conclusión: Con RAG la respuesta es más cercana a la respuesta que daría una persona.